# 01 Exact enumeration

This notebook simulates every staffing matrix of every setting on common random numbers and saves the results to `clinic2026_paper_v3/data/enum/`. The analysis notebooks (02 to 05) read these files and do not simulate the enumeration again.

**Jobs, in the order they run**

| Jobs | Staffings each | Days | Purpose |
|---|---|---|---|
| 8 settings with 2 blocks | 6,561 | 200 | Main sample |
| 8 settings with 3 blocks | 531,441 | 200 | Main sample |
| 8 settings with 2 blocks, 4 independent samples | 6,561 | 200 | Sampling checks |
| 8 settings with 2 blocks, reference sample | 6,561 | 1,000 | Sampling checks |

**How to run it**

1. Run the setup cell, then the run cell.
2. The run cell works through the jobs in chunks and saves every chunk to Drive as soon as it is done. It stops by itself after `MAX_MINUTES` (50 by default).
3. Run it again, in the same session or a new one, until it reports that all jobs are finished. Finished chunks are never recomputed, so a disconnection loses at most one chunk, about a minute of work.
4. When a job finishes, its chunks are merged into one compressed file.

**Run time:** notebook 00 printed an estimate for your runtime. The three-block main samples take most of it. Notebooks 02 to 04 can run as soon as the jobs they need are finished, because each of them skips the settings that are not enumerated yet.

In [ ]:
# Setup: storage folder, numba and the core module written by notebook 00
import os, sys, shutil, subprocess, time, math
import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/My Drive/clinic2026_final'
else:
    BASE = os.environ.get('CLINIC_BASE', os.path.abspath('clinic2026_final'))
try:
    import numba  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'], check=True)

CORE = os.path.join(BASE, 'core', 'clinic_core.py')
if not os.path.exists(CORE):
    raise FileNotFoundError('Run notebook 00 first; it writes the core module to ' + CORE)
LOCAL = '/content/clinic_core_local' if IN_COLAB else os.path.join(BASE, 'core')
os.makedirs(LOCAL, exist_ok=True)
if os.path.abspath(LOCAL) != os.path.abspath(os.path.dirname(CORE)):
    shutil.copy(CORE, LOCAL)          # numba caches its compiled code next to the module, so keep it on local disk
sys.path.insert(0, LOCAL)
import clinic_core as cc
P = cc.make_paths(BASE)
os.makedirs(os.path.join(P['results'], 'parts'), exist_ok=True)
print('Project folder:', BASE)
print('numba available:', cc.HAVE_NUMBA)

Mounted at /content/drive
Project folder: /content/drive/My Drive/clinic2026_paper_v3
numba available: True


In [ ]:
# The enumeration jobs and their status
jobs = cc.enumeration_jobs()
status = cc.enumeration_status(P['data'], jobs)
print(status.status.value_counts().to_string())
status

status
pending    56


,job,staffings,days,status
0,B2-PAR1.0-moderate_n200_s0,6561,200,pending
1,B2-PAR1.0-high_n200_s0,6561,200,pending
2,B2-PAR1.3-moderate_n200_s0,6561,200,pending
3,B2-PAR1.3-high_n200_s0,6561,200,pending
4,B2-PAR1.6-moderate_n200_s0,6561,200,pending
5,B2-PAR1.6-high_n200_s0,6561,200,pending
6,B2-PAR2.0-moderate_n200_s0,6561,200,pending
7,B2-PAR2.0-high_n200_s0,6561,200,pending
8,B3-PAR1.0-moderate_n200_s0,531441,200,pending
9,B3-PAR1.0-high_n200_s0,531441,200,pending


In [ ]:
# Run the queue for at most MAX_MINUTES, then stop; run this cell again to continue
MAX_MINUTES = 50
done = cc.run_enumeration_queue(P['data'], jobs, max_minutes=MAX_MINUTES)
print('All jobs finished.' if done else 'Not finished yet: run this cell again.')

  B3-PAR2.0-high_n200_s0: rows 500,000 to 519,999 of 531,441 in 59s
  B3-PAR2.0-high_n200_s0: rows 520,000 to 531,440 of 531,441 in 34s
Finished B3-PAR2.0-high_n200_s0 (531,441 staffings, 200 days). Jobs finished: 16 of 56.
  B2-PAR1.0-moderate_n200_s10000: rows 0 to 6,560 of 6,561 in 18s
Finished B2-PAR1.0-moderate_n200_s10000 (6,561 staffings, 200 days). Jobs finished: 17 of 56.
  B2-PAR1.0-moderate_n200_s20000: rows 0 to 6,560 of 6,561 in 19s
Finished B2-PAR1.0-moderate_n200_s20000 (6,561 staffings, 200 days). Jobs finished: 18 of 56.
  B2-PAR1.0-moderate_n200_s30000: rows 0 to 6,560 of 6,561 in 17s
Finished B2-PAR1.0-moderate_n200_s30000 (6,561 staffings, 200 days). Jobs finished: 19 of 56.
  B2-PAR1.0-moderate_n200_s40000: rows 0 to 6,560 of 6,561 in 17s
Finished B2-PAR1.0-moderate_n200_s40000 (6,561 staffings, 200 days). Jobs finished: 20 of 56.
  B2-PAR1.0-moderate_n1000_s100000: rows 0 to 3,999 of 6,561 in 57s
  B2-PAR1.0-moderate_n1000_s100000: rows 4,000 to 6,560 of 6,561 in 

In [ ]:
# After all jobs are finished: a spot check of each file against a fresh simulation of a few random plans
if cc.enumeration_status(P['data'], jobs).status.eq('done').all():
    rng = np.random.default_rng(2026)
    rows_out = []
    for s, n, off in jobs:
        df = cc.load_table(P['data'], s, n, off)
        rows = np.sort(rng.choice(len(df), size=5, replace=False))
        X = cc.Evaluator(s, n, off).evaluate_rows(rows)
        ref = X[:, [cc.METRIC_COLS.index(k) for k in cc.STORED]]
        rows_out.append({'job': cc.enum_name(s, n, off), 'identical': bool(np.array_equal(ref, df.loc[rows, list(cc.STORED)].values)),
                         'feasible staffings': int(df.feasible.sum())})
    chk = pd.DataFrame(rows_out)
    chk.to_csv(os.path.join(P['results'], 'enumeration_check.csv'), index=False)
    print(chk.to_string(index=False))
    assert chk.identical.all()
else:
    print('Some jobs are not finished yet.')

                             job  identical  feasible staffings
      B2-PAR1.0-moderate_n200_s0       True                 303
          B2-PAR1.0-high_n200_s0       True                 174
      B2-PAR1.3-moderate_n200_s0       True                 225
          B2-PAR1.3-high_n200_s0       True                  96
      B2-PAR1.6-moderate_n200_s0       True                 126
          B2-PAR1.6-high_n200_s0       True                  68
      B2-PAR2.0-moderate_n200_s0       True                  40
          B2-PAR2.0-high_n200_s0       True                  15
      B3-PAR1.0-moderate_n200_s0       True                8509
          B3-PAR1.0-high_n200_s0       True                3748
      B3-PAR1.3-moderate_n200_s0       True                7045
          B3-PAR1.3-high_n200_s0       True                2823
      B3-PAR1.6-moderate_n200_s0       True                3925
          B3-PAR1.6-high_n200_s0       True                1257
      B3-PAR2.0-moderate_n200_s0       T